### 1. Import Libraries

In [11]:
import json
from pathlib import Path
import pandas as pd

### 2. Set the ICSA root directory

In [12]:
current_dir = Path.cwd().resolve()
BASE_DIR = next(
    (path for path in [current_dir, *current_dir.parents] if (path / 'data').exists()),
    None
)

if BASE_DIR is None:
    raise FileNotFoundError("Could not find the project root containing the 'data' directory.")

ICSA_ROOT = BASE_DIR / 'data' / 'raw' / 'icsa'
TARGET_YEARS = list(range(2016, 2025))

print(f'[INFO] BASE_DIR: {BASE_DIR}')
print(f'[INFO] ICSA_ROOT: {ICSA_ROOT}')
print(f'[INFO] TARGET_YEARS: {TARGET_YEARS}')
print(f'[INFO] Available year directories: {[p.name for p in ICSA_ROOT.iterdir() if p.is_dir()]}')

[INFO] BASE_DIR: C:\Workspace\icsa-threat-kg
[INFO] ICSA_ROOT: C:\Workspace\icsa-threat-kg\data\raw\icsa
[INFO] TARGET_YEARS: [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
[INFO] Available year directories: ['2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025', '2026']


### 3. Recursively parse product_tree to build product_id mapping

In [17]:
def build_product_map(product_tree):
    product_map = {}

    def traverse(node, vendor=None, product_name=None, path_labels=None):
        if path_labels is None:
            path_labels = []

        if not isinstance(node, dict):
            return

        current_vendor = vendor
        current_product_name = product_name
        current_path_labels = list(path_labels)

        node_name = node.get('name')
        node_category = node.get('category')

        if isinstance(node_name, str):
            current_path_labels.append(node_name)

        if node_category == 'vendor':
            current_vendor = node_name
        elif node_category == 'product_name':
            current_product_name = node_name

        product = node.get('product')
        if isinstance(product, dict):
            product_id = product.get('product_id')
            full_name = product.get('name')

            if product_id:
                product_map[product_id] = {
                    'vendor': current_vendor,
                    'product_name': current_product_name,
                    'full_name': full_name,
                    'path_labels': current_path_labels,
                }

        for child in node.get('branches', []):
            traverse(
                child,
                vendor=current_vendor,
                product_name=current_product_name,
                path_labels=current_path_labels,
            )

    for branch in product_tree.get('branches', []):
        traverse(branch)

    return product_map

### 4. Extract minimal fields from one advisory JSON

In [20]:
def extract_icsa_min_fields(json_path):
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    document = data.get('document', {})
    tracking = document.get('tracking', {})
    advisory_id = tracking.get('id')

    if advisory_id is None:
        return None

    product_tree = data.get('product_tree', {})
    product_map = build_product_map(product_tree)

    cve_set = set()
    affected_product_ids = set()

    for vuln in data.get('vulnerabilities', []):
        cve_id = vuln.get('cve')
        if cve_id:
            cve_set.add(cve_id)

        product_status = vuln.get('product_status', {})
        for pid in product_status.get('known_affected', []):
            affected_product_ids.add(pid)

    affected_product_records = []
    for pid in sorted(affected_product_ids):
        product_info = product_map.get(pid, {})
        affected_product_records.append({
            'product_id': pid,
            'vendor': product_info.get('vendor'),
            'product_name': product_info.get('product_name'),
            'full_name': product_info.get('full_name'),
        })

    return {
        'advisory_id': advisory_id,
        'cve_list': sorted(cve_set),
        'affected_product_records': affected_product_records,
        'source_file': str(json_path),
    }

### 5. Load all advisory files from 2016-2024

In [21]:
icsa_records = []
num_failed = 0

for year in TARGET_YEARS:
    year_dir = ICSA_ROOT / str(year)
    if not year_dir.exists():
        print(f'[WARN] Directory not found: {year_dir}')
        continue

    json_files = sorted(year_dir.glob('icsa-*.json'))
    print(f'[INFO] Year {year}: {len(json_files)} ICSA JSON files found')

    for json_file in json_files:
        try:
            record = extract_icsa_min_fields(json_file)
            if record is not None:
                icsa_records.append(record)
        except Exception as e:
            num_failed += 1
            print(f'[ERROR] Failed to parse {json_file.name}: {e}')

print(f'\n[DONE] Parsed {len(icsa_records)} ICSA records successfully')
print(f'[INFO] Failed files: {num_failed}')

[INFO] Year 2016: 126 ICSA JSON files found
[INFO] Year 2017: 171 ICSA JSON files found
[INFO] Year 2018: 192 ICSA JSON files found
[INFO] Year 2019: 204 ICSA JSON files found
[INFO] Year 2020: 225 ICSA JSON files found
[INFO] Year 2021: 369 ICSA JSON files found
[INFO] Year 2022: 370 ICSA JSON files found
[INFO] Year 2023: 370 ICSA JSON files found
[INFO] Year 2024: 409 ICSA JSON files found

[DONE] Parsed 2436 ICSA records successfully
[INFO] Failed files: 0


### 6. Convert to DataFrame for quick inspection

In [22]:
icsa_df = pd.DataFrame({
    'advisory_id': [r['advisory_id'] for r in icsa_records],
    'num_cves': [len(r['cve_list']) for r in icsa_records],
    'num_affected_products': [len(r['affected_product_records']) for r in icsa_records],
    'source_file': [r['source_file'] for r in icsa_records],
})

icsa_df['year'] = icsa_df['source_file'].apply(lambda x: Path(x).parent.name)
icsa_df = icsa_df[['year', 'advisory_id', 'num_cves', 'num_affected_products', 'source_file']]

print(f'\n[DONE] Parsed {len(icsa_records)} ICSA advisories')
display(icsa_df.head())


[DONE] Parsed 2436 ICSA advisories


,year,advisory_id,num_cves,num_affected_products,source_file
0,2016,ICSA-16-014-01,15,1,C:\Workspace\icsa-threat-kg\data\raw\icsa\2016...
1,2016,ICSA-16-019-01,1,2,C:\Workspace\icsa-threat-kg\data\raw\icsa\2016...
2,2016,ICSA-16-021-01,1,1,C:\Workspace\icsa-threat-kg\data\raw\icsa\2016...
3,2016,ICSA-16-026-01,1,1,C:\Workspace\icsa-threat-kg\data\raw\icsa\2016...
4,2016,ICSA-16-026-02,1,8,C:\Workspace\icsa-threat-kg\data\raw\icsa\2016...


### 7. Example: inspect one advisory

In [23]:
if len(icsa_records) > 0:
    sample = icsa_records[0]
    print('\n[Sample Advisory]')
    print('advisory_id:', sample['advisory_id'])
    print('num_cves:', len(sample['cve_list']))
    print('num_affected_products:', len(sample['affected_product_records']))
    print('cve_list:', sample['cve_list'][:5], '...')
    print('affected_product_records:', sample['affected_product_records'][:3])


[Sample Advisory]
advisory_id: ICSA-16-014-01
num_cves: 15
num_affected_products: 1
cve_list: ['CVE-2015-3943', 'CVE-2015-3946', 'CVE-2015-3947', 'CVE-2015-3948', 'CVE-2015-6467'] ...
affected_product_records: [{'product_id': 'CSAFPID-0001', 'vendor': 'Advantech', 'product_name': 'WebAccess', 'full_name': 'Advantech WebAccess: <=8.0'}]


### 8. Generate ICSA–CVE Triples and Prepare Product Records for CPE Matching

In [24]:
import re
import pandas as pd


def normalize_text(text):
    if text is None:
        return None

    text = text.lower().strip()
    text = text.replace('&', ' and ')
    text = re.sub(r'\(.*?\)', ' ', text)         # remove parentheses content
    text = re.sub(r'vers:[^ ]+', ' ', text)      # remove CSAF version tokens
    text = re.sub(r'[^a-z0-9]+', '_', text)      # keep alnum, replace others with _
    text = re.sub(r'_+', '_', text).strip('_')   # collapse underscores

    return text if text else None


icsa_cve_triples = []
icsa_product_rows = []

for record in icsa_records:
    advisory_id = record['advisory_id']

    # 1. ICSA -> CVE triples
    for cve_id in record['cve_list']:
        icsa_cve_triples.append((advisory_id, 'includesCVE', cve_id))

    # 2. ICSA -> raw product rows (for later CPE matching)
    for product in record['affected_product_records']:
        vendor = product.get('vendor')
        product_name = product.get('product_name')
        full_name = product.get('full_name')

        icsa_product_rows.append({
            'advisory_id': advisory_id,
            'product_id': product.get('product_id'),
            'vendor': vendor,
            'product_name': product_name,
            'full_name': full_name,
            'vendor_norm': normalize_text(vendor),
            'product_name_norm': normalize_text(product_name),
            'full_name_norm': normalize_text(full_name),
        })

icsa_cve_df = pd.DataFrame(icsa_cve_triples, columns=['head', 'relation', 'tail'])
icsa_product_df = pd.DataFrame(icsa_product_rows)

print(f'[DONE] ICSA-CVE triples: {len(icsa_cve_df)}')
print(f'[DONE] ICSA raw product rows: {len(icsa_product_df)}')

display(icsa_cve_df.head())
display(icsa_product_df.head())

[DONE] ICSA-CVE triples: 8704
[DONE] ICSA raw product rows: 17721


,head,relation,tail
0,ICSA-16-014-01,includesCVE,CVE-2015-3943
1,ICSA-16-014-01,includesCVE,CVE-2015-3946
2,ICSA-16-014-01,includesCVE,CVE-2015-3947
3,ICSA-16-014-01,includesCVE,CVE-2015-3948
4,ICSA-16-014-01,includesCVE,CVE-2015-6467


,advisory_id,product_id,vendor,product_name,full_name,vendor_norm,product_name_norm,full_name_norm
0,ICSA-16-014-01,CSAFPID-0001,Advantech,WebAccess,Advantech WebAccess: <=8.0,advantech,webaccess,advantech_webaccess_8_0
1,ICSA-16-019-01,CSAFPID-0001,Siemens,OZW672,Siemens OZW672: <V6.00,siemens,ozw672,siemens_ozw672_v6_00
2,ICSA-16-019-01,CSAFPID-0002,Siemens,OZW772,Siemens OZW772: <V6.00,siemens,ozw772,siemens_ozw772_v6_00
3,ICSA-16-021-01,CSAFPID-0001,CAREL,PlantVisorEnhanced,CAREL PlantVisorEnhanced: vers:all/*,carel,plantvisorenhanced,carel_plantvisorenhanced
4,ICSA-16-026-01,CSAFPID-0001,MICROSYS,PROMOTIC,MICROSYS PROMOTIC: <8.3.11,microsys,promotic,microsys_promotic_8_3_11


### 9. Match ICSA Product Records to Versionless CPE Candidates

In [43]:
try:
    from rapidfuzz import fuzz

    def similarity_score(a, b):
        a = a if isinstance(a, str) else ''
        b = b if isinstance(b, str) else ''
        return fuzz.ratio(a, b)
except ImportError:
    from difflib import SequenceMatcher

    def similarity_score(a, b):
        a = a if isinstance(a, str) else ''
        b = b if isinstance(b, str) else ''
        return SequenceMatcher(None, a, b).ratio() * 100

In [44]:
# ---------------------------------------------------------
# 1. Set the versionless CPE CSV path
# ---------------------------------------------------------
VERSIONLESS_CPE_PATH = BASE_DIR / 'data' / 'processed' / 'cpe' / 'cpe_ignore_version.csv'

FUZZY_THRESHOLD = 90
VENDOR_THRESHOLD = 95

In [45]:
# ---------------------------------------------------------
# 2. Normalization helpers
# ---------------------------------------------------------
def normalize_text(text):
    if pd.isna(text) or text is None:
        return None

    text = str(text).lower().strip()
    text = text.replace('&', ' and ')
    text = re.sub(r'\(.*?\)', ' ', text)          # remove parentheses content
    text = re.sub(r'vers:[^ ]+', ' ', text)       # remove CSAF version tokens
    text = re.sub(r'[^a-z0-9]+', '_', text)       # keep alnum only
    text = re.sub(r'_+', '_', text).strip('_')    # collapse underscores

    return text if text else None


def is_contains_match(a, b):
    if not isinstance(a, str) or not isinstance(b, str):
        return False
    if not a or not b:
        return False
    return a in b or b in a

In [46]:
# ---------------------------------------------------------
# 3. Load the versionless CPE candidate table
# ---------------------------------------------------------
cpe_df = pd.read_csv(VERSIONLESS_CPE_PATH).copy()

cpe_df['vendor_norm'] = cpe_df['vendor'].apply(normalize_text)
cpe_df['product_norm'] = cpe_df['product'].apply(normalize_text)
cpe_df['composite_norm'] = (
    cpe_df['vendor'].fillna('').astype(str) + ' ' +
    cpe_df['product'].fillna('').astype(str)
).apply(normalize_text)

# Optional: remove duplicate versionless CPE rows
cpe_df = cpe_df.drop_duplicates(subset=['versionless_cpe']).reset_index(drop=True)

print(f'[INFO] Loaded {len(cpe_df)} unique versionless CPE candidates')

[INFO] Loaded 151307 unique versionless CPE candidates


In [47]:
# ---------------------------------------------------------
# 4. Vendor candidate selection
# ---------------------------------------------------------
unique_vendors = cpe_df['vendor_norm'].dropna().unique().tolist()


def select_vendor_subset(icsa_vendor_norm):
    if not icsa_vendor_norm:
        return cpe_df.iloc[0:0].copy(), None, 0, 'none'

    # Exact vendor match
    exact_subset = cpe_df[cpe_df['vendor_norm'] == icsa_vendor_norm]
    if not exact_subset.empty:
        return exact_subset, icsa_vendor_norm, 100, 'exact'

    # Fuzzy vendor fallback
    best_vendor = None
    best_score = -1

    for vendor_norm in unique_vendors:
        score = similarity_score(icsa_vendor_norm, vendor_norm)
        if score > best_score:
            best_score = score
            best_vendor = vendor_norm

    if best_vendor is not None and best_score >= VENDOR_THRESHOLD:
        fuzzy_subset = cpe_df[cpe_df['vendor_norm'] == best_vendor]
        return fuzzy_subset, best_vendor, best_score, 'fuzzy'

    return cpe_df.iloc[0:0].copy(), None, best_score, 'unmatched'

In [48]:
# ---------------------------------------------------------
# 5. Product matching within the selected vendor subset
# ---------------------------------------------------------
def match_one_icsa_product(row):
    icsa_vendor_norm = row.get('vendor_norm')
    icsa_product_norm = row.get('product_name_norm')
    icsa_full_name_norm = row.get('full_name_norm')

    vendor_subset, matched_vendor_norm, vendor_score, vendor_match_type = select_vendor_subset(icsa_vendor_norm)

    if vendor_subset.empty:
        return {
            'matched': False,
            'match_type': 'unmatched',
            'match_score': 0,
            'vendor_match_type': vendor_match_type,
            'vendor_match_score': vendor_score,
            'matched_vendor': None,
            'matched_product': None,
            'matched_cpe': None,
            'matched_part': None,
            'matched_target_sw': None,
            'matched_target_hw': None,
            'candidate_count': 0,
        }

    # 1) Exact product match
    exact_matches = vendor_subset[vendor_subset['product_norm'] == icsa_product_norm]
    if not exact_matches.empty:
        best = exact_matches.sort_values(by='size', ascending=False).iloc[0]
        return {
            'matched': True,
            'match_type': 'exact',
            'match_score': 100,
            'vendor_match_type': vendor_match_type,
            'vendor_match_score': vendor_score,
            'matched_vendor': best['vendor'],
            'matched_product': best['product'],
            'matched_cpe': best['versionless_cpe'],
            'matched_part': best['part'],
            'matched_target_sw': best['target_sw'],
            'matched_target_hw': best['target_hw'],
            'candidate_count': len(vendor_subset),
        }

    # 2) Contains match
    contains_subset = vendor_subset[
        vendor_subset['product_norm'].apply(lambda x: is_contains_match(icsa_product_norm, x))
    ]
    if not contains_subset.empty:
        best = contains_subset.sort_values(by='size', ascending=False).iloc[0]
        return {
            'matched': True,
            'match_type': 'contains',
            'match_score': 95,
            'vendor_match_type': vendor_match_type,
            'vendor_match_score': vendor_score,
            'matched_vendor': best['vendor'],
            'matched_product': best['product'],
            'matched_cpe': best['versionless_cpe'],
            'matched_part': best['part'],
            'matched_target_sw': best['target_sw'],
            'matched_target_hw': best['target_hw'],
            'candidate_count': len(vendor_subset),
        }

    # 3) Fuzzy product match
    scored_candidates = []
    for _, cand in vendor_subset.iterrows():
        product_score = similarity_score(icsa_product_norm, cand['product_norm'])
        composite_score = similarity_score(icsa_full_name_norm, cand['composite_norm'])
        final_score = max(product_score, composite_score)

        scored_candidates.append({
            'versionless_cpe': cand['versionless_cpe'],
            'vendor': cand['vendor'],
            'product': cand['product'],
            'part': cand['part'],
            'target_sw': cand['target_sw'],
            'target_hw': cand['target_hw'],
            'score': final_score,
            'size': cand['size'],
        })

    scored_df = pd.DataFrame(scored_candidates).sort_values(
        by=['score', 'size'],
        ascending=[False, False]
    )

    if scored_df.empty:
        return {
            'matched': False,
            'match_type': 'unmatched',
            'match_score': 0,
            'vendor_match_type': vendor_match_type,
            'vendor_match_score': vendor_score,
            'matched_vendor': None,
            'matched_product': None,
            'matched_cpe': None,
            'matched_part': None,
            'matched_target_sw': None,
            'matched_target_hw': None,
            'candidate_count': len(vendor_subset),
        }

    best = scored_df.iloc[0]

    if best['score'] >= FUZZY_THRESHOLD:
        return {
            'matched': True,
            'match_type': 'fuzzy',
            'match_score': best['score'],
            'vendor_match_type': vendor_match_type,
            'vendor_match_score': vendor_score,
            'matched_vendor': best['vendor'],
            'matched_product': best['product'],
            'matched_cpe': best['versionless_cpe'],
            'matched_part': best['part'],
            'matched_target_sw': best['target_sw'],
            'matched_target_hw': best['target_hw'],
            'candidate_count': len(vendor_subset),
        }

    return {
        'matched': False,
        'match_type': 'unmatched',
        'match_score': best['score'],
        'vendor_match_type': vendor_match_type,
        'vendor_match_score': vendor_score,
        'matched_vendor': best['vendor'],
        'matched_product': best['product'],
        'matched_cpe': None,
        'matched_part': best['part'],
        'matched_target_sw': best['target_sw'],
        'matched_target_hw': best['target_hw'],
        'candidate_count': len(vendor_subset),
    }

In [50]:
# ---------------------------------------------------------
# 6. Apply matching to all ICSA product rows
# ---------------------------------------------------------
match_results = []

for _, row in icsa_product_df.iterrows():
    result = match_one_icsa_product(row)

    match_results.append({
        'advisory_id': row['advisory_id'],
        'product_id': row['product_id'],
        'vendor': row['vendor'],
        'product_name': row['product_name'],
        'full_name': row['full_name'],
        'vendor_norm': row['vendor_norm'],
        'product_name_norm': row['product_name_norm'],
        'full_name_norm': row['full_name_norm'],
        **result
    })

icsa_product_match_df = pd.DataFrame(match_results)

print(f'[DONE] Processed {len(icsa_product_match_df)} ICSA product rows')
print('\n[Match Type Counts]')
print(icsa_product_match_df['match_type'].value_counts(dropna=False))

display(icsa_product_match_df.head())

[DONE] Processed 17721 ICSA product rows

[Match Type Counts]
match_type
exact        8906
contains     4259
unmatched    4221
fuzzy         335
Name: count, dtype: int64


,advisory_id,product_id,vendor,product_name,full_name,vendor_norm,product_name_norm,full_name_norm,matched,match_type,match_score,vendor_match_type,vendor_match_score,matched_vendor,matched_product,matched_cpe,matched_part,matched_target_sw,matched_target_hw,candidate_count
0,ICSA-16-014-01,CSAFPID-0001,Advantech,WebAccess,Advantech WebAccess: <=8.0,advantech,webaccess,advantech_webaccess_8_0,True,exact,100.0,exact,100.0,advantech,webaccess,cpe:a:advantech:webaccess:*:*,a,*,*,55
1,ICSA-16-019-01,CSAFPID-0001,Siemens,OZW672,Siemens OZW672: <V6.00,siemens,ozw672,siemens_ozw672_v6_00,True,exact,100.0,exact,100.0,siemens,ozw672,cpe:h:siemens:ozw672:*:*,h,*,*,3992
2,ICSA-16-019-01,CSAFPID-0002,Siemens,OZW772,Siemens OZW772: <V6.00,siemens,ozw772,siemens_ozw772_v6_00,True,exact,100.0,exact,100.0,siemens,ozw772,cpe:h:siemens:ozw772:*:*,h,*,*,3992
3,ICSA-16-021-01,CSAFPID-0001,CAREL,PlantVisorEnhanced,CAREL PlantVisorEnhanced: vers:all/*,carel,plantvisorenhanced,carel_plantvisorenhanced,True,exact,100.0,exact,100.0,carel,plantvisorenhanced,cpe:a:carel:plantvisorenhanced:*:*,a,*,*,12
4,ICSA-16-026-01,CSAFPID-0001,MICROSYS,PROMOTIC,MICROSYS PROMOTIC: <8.3.11,microsys,promotic,microsys_promotic_8_3_11,True,exact,100.0,exact,100.0,microsys,promotic,cpe:a:microsys:promotic:*:*,a,*,*,1


In [51]:
# ---------------------------------------------------------
# 7. Keep only accepted matches and create ICSA -> CPE triples
# ---------------------------------------------------------
accepted_match_types = {'exact', 'contains', 'fuzzy'}

accepted_icsa_cpe_df = icsa_product_match_df[
    (icsa_product_match_df['matched'] == True) &
    (icsa_product_match_df['match_type'].isin(accepted_match_types))
].copy()

icsa_cpe_df = (
    accepted_icsa_cpe_df[['advisory_id', 'matched_cpe']]
    .dropna()
    .drop_duplicates()
    .rename(columns={'advisory_id': 'head', 'matched_cpe': 'tail'})
)

icsa_cpe_df['relation'] = 'affectsCPE'
icsa_cpe_df = icsa_cpe_df[['head', 'relation', 'tail']]

print(f'\n[DONE] Accepted ICSA-CPE triples: {len(icsa_cpe_df)}')
display(icsa_cpe_df.head())


[DONE] Accepted ICSA-CPE triples: 8367


,head,relation,tail
0,ICSA-16-014-01,affectsCPE,cpe:a:advantech:webaccess:*:*
1,ICSA-16-019-01,affectsCPE,cpe:h:siemens:ozw672:*:*
2,ICSA-16-019-01,affectsCPE,cpe:h:siemens:ozw772:*:*
3,ICSA-16-021-01,affectsCPE,cpe:a:carel:plantvisorenhanced:*:*
4,ICSA-16-026-01,affectsCPE,cpe:a:microsys:promotic:*:*


In [52]:
match_summary = (
    icsa_product_match_df['match_type']
    .value_counts(dropna=False)
    .rename_axis('match_type')
    .reset_index(name='count')
)

match_summary['ratio'] = match_summary['count'] / len(icsa_product_match_df)
match_summary['ratio'] = match_summary['ratio'].map(lambda x: f'{x:.3%}')
match_summary

,match_type,count,ratio
0,exact,8906,50.257%
1,contains,4259,24.034%
2,unmatched,4221,23.819%
3,fuzzy,335,1.890%


In [55]:
def parse_icsa_sort_key(icsa_id):
    """
    Convert an ICSA ID such as 'ICSA-16-348-03' into a sortable tuple.
    """
    if not isinstance(icsa_id, str):
        return (9999, 9999, 9999)

    match = re.match(r'^ICSA-(\d{2})-(\d+)-(\d+)$', icsa_id)
    if match:
        year, advisory_num, sub_num = match.groups()
        return (int(year), int(advisory_num), int(sub_num))

    return (9999, 9999, 9999)

In [57]:
# ---------------------------------------------------------
# 1. Combine ICSA-CVE and ICSA-CPE triples
# ---------------------------------------------------------
icsa_triples_df = pd.concat(
    [icsa_cve_df.copy(), icsa_cpe_df.copy()],
    ignore_index=True
).drop_duplicates()

# Optional: relation order
relation_order = ['includesCVE', 'affectsCPE']
icsa_triples_df['relation'] = pd.Categorical(
    icsa_triples_df['relation'],
    categories=relation_order,
    ordered=True
)

In [58]:
# ---------------------------------------------------------
# 2. Sort by ICSA ID, relation, and tail
# ---------------------------------------------------------
sort_keys = icsa_triples_df['head'].apply(parse_icsa_sort_key)
icsa_triples_df['sort_year'] = sort_keys.apply(lambda x: x[0])
icsa_triples_df['sort_adv'] = sort_keys.apply(lambda x: x[1])
icsa_triples_df['sort_sub'] = sort_keys.apply(lambda x: x[2])

icsa_triples_df = icsa_triples_df.sort_values(
    by=['sort_year', 'sort_adv', 'sort_sub', 'relation', 'tail'],
    ascending=[True, True, True, True, True]
).reset_index(drop=True)

# Remove helper columns
icsa_triples_df = icsa_triples_df[['head', 'relation', 'tail']]

print(f'[DONE] Total ICSA triples: {len(icsa_triples_df)}')
display(icsa_triples_df.head(20))

[DONE] Total ICSA triples: 17071


,head,relation,tail
0,ICSA-16-014-01,includesCVE,CVE-2015-3943
1,ICSA-16-014-01,includesCVE,CVE-2015-3946
2,ICSA-16-014-01,includesCVE,CVE-2015-3947
3,ICSA-16-014-01,includesCVE,CVE-2015-3948
4,ICSA-16-014-01,includesCVE,CVE-2015-6467
5,ICSA-16-014-01,includesCVE,CVE-2016-0851
6,ICSA-16-014-01,includesCVE,CVE-2016-0852
7,ICSA-16-014-01,includesCVE,CVE-2016-0853
8,ICSA-16-014-01,includesCVE,CVE-2016-0854
9,ICSA-16-014-01,includesCVE,CVE-2016-0855


In [60]:
# ---------------------------------------------------------
# 3. Save the final triple file
# ---------------------------------------------------------
SAVE_DIR = BASE_DIR / 'data' / 'processed' / 'kg' / '2024_12'
SAVE_DIR.mkdir(parents=True, exist_ok=True)

icsa_triple_path = SAVE_DIR / 'icsa_threat_kg.csv'
icsa_triples_df.to_csv(icsa_triple_path, index=False)

print(f'[DONE] Saved ICSA triples to: {icsa_triple_path}')

[DONE] Saved ICSA triples to: C:\Workspace\icsa-threat-kg\data\processed\kg\2024_12\icsa_threat_kg.csv


In [61]:
# ---------------------------------------------------------
# 4. Optional: Save grouped summary by ICSA advisory
# ---------------------------------------------------------
icsa_group_summary_df = (
    icsa_triples_df.groupby('head')
    .agg(
        num_triples=('tail', 'count'),
        num_cves=('relation', lambda x: (x == 'includesCVE').sum()),
        num_cpes=('relation', lambda x: (x == 'affectsCPE').sum())
    )
    .reset_index()
)

icsa_group_summary_path = SAVE_DIR / 'icsa_group_summary.csv'
icsa_group_summary_df.to_csv(icsa_group_summary_path, index=False)

print(f'[DONE] Saved ICSA group summary to: {icsa_group_summary_path}')
display(icsa_group_summary_df.head(20))

[DONE] Saved ICSA group summary to: C:\Workspace\icsa-threat-kg\data\processed\kg\2024_12\icsa_group_summary.csv


,head,num_triples,num_cves,num_cpes
0,ICSA-16-014-01,16,15,1
1,ICSA-16-019-01,3,1,2
2,ICSA-16-021-01,2,1,1
3,ICSA-16-026-01,2,1,1
4,ICSA-16-026-02,9,1,8
5,ICSA-16-028-01A,2,1,1
6,ICSA-16-033-01,4,3,1
7,ICSA-16-033-02,2,2,0
8,ICSA-16-040-01,4,4,0
9,ICSA-16-040-02,3,2,1
